# Practical P18: Memory-Enabled Chatbot with LangChain & Google Gemini
**Course**: PGDCA — Hands-On Large Language Models  
**Unit 3**: LLM Frameworks for Application Development  
**Syllabus Topic**: 3.1 LangChain framework: chains, memory, output parsers  
**Model Provider**: Google Gemini (`gemini-1.5-flash`) via `langchain-google-genai`  
**Learning Outcome**: Understand how conversational state works in stateless LLMs, implement classic conversation buffer memory, and construct stateful multi-turn conversational agents with session isolation using modern LCEL `RunnableWithMessageHistory` and real Google Gemini calls.

## Part 1: Theoretical Foundations — Conversational Memory

### 1.1 The Stateless Nature of LLMs
LLMs operate over stateless HTTP request/response lifecycles. When you send a message to Google Gemini, the server processes it and closes the connection. The model holds **no persistent memory** of previous queries:
* *Turn 1*: "Hi, my name is Alice." $\rightarrow$ Model replies: "Hello Alice!"
* *Turn 2*: "What is my name?" $\rightarrow$ Without memory injection, the model has no knowledge of Alice!

### 1.2 How Memory Works Under the Hood
To create the illusion of memory, the application must:
1. Capture the user input and the model output on each turn.
2. Store the interaction history in a data structure or database.
3. Automatically prepend or inject prior messages into the prompt payload on subsequent turns.

```mermaid
graph TD
    subgraph Conversation Turn
        User["User: 'What is my name?'"] --> Inject["Inject Session History"]
        History[("Stored Messages: ['User: Alice', 'AI: Hi Alice']")] --> Inject
        Inject --> Prompt["Combined Prompt with History"]
        Prompt --> Gemini["Google Gemini (gemini-1.5-flash)"]
        Gemini --> Reply["AI: 'Your name is Alice.'"]
        Reply --> Save["Save Turn to History"]
        User --> Save
        Save --> History
    end
```

### 1.3 Memory Strategies in LangChain
* **Buffer Memory**: Stores all raw messages. 100% accurate, but context tokens grow linearly.
* **Window Memory**: Keeps only the last $K$ turns (sliding window), keeping token usage bounded.
* **Summary Memory**: Uses Gemini to maintain an evolving summary narrative of old turns.
* **Modern LCEL Memory**: Separates the storage backend (`InMemoryChatMessageHistory`) from the execution logic (`RunnableWithMessageHistory`), supporting multi-user `session_id` isolation.

## Part 2: Working with LangChain Message Histories and Buffer Formatting

LangChain Core provides `InMemoryChatMessageHistory` and message formatting utilities like `get_buffer_string` to inspect raw conversational memory.

In [1]:
# Part 2 Code: InMemoryChatMessageHistory and get_buffer_string
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import get_buffer_string

# 1. Initialize in-memory chat message history
chat_history = InMemoryChatMessageHistory()

# 2. Add User and AI messages across 2 conversation turns
chat_history.add_user_message("Hello! My name is Alice and I am a PGDCA student.")
chat_history.add_ai_message("Hello Alice! Welcome to the PGDCA course. How can I help you today?")

chat_history.add_user_message("I am working on Unit 3 practicals on LangChain and Gemini.")
chat_history.add_ai_message("Great! Unit 3 covers LCEL chains, conversational memory, output parsers, and vector stores.")

# 3. Format buffer into readable text
formatted_buffer = get_buffer_string(chat_history.messages)

print("=" * 60)
print("🧠 CHAT BUFFER FORMATTED STRING:")
print("=" * 60)
print(formatted_buffer)
print("-" * 60)
print(f"Total stored messages in history: {len(chat_history.messages)}")

🧠 CHAT BUFFER FORMATTED STRING:
Human: Hello! My name is Alice and I am a PGDCA student.
AI: Hello Alice! Welcome to the PGDCA course. How can I help you today?
Human: I am working on Unit 3 practicals on LangChain and Gemini.
AI: Great! Unit 3 covers LCEL chains, conversational memory, output parsers, and vector stores.
------------------------------------------------------------
Total stored messages in history: 4


## Part 3: Modern LCEL Stateful Chatbot with Real Google Gemini

In modern LangChain, memory is decoupled from the chain definition.
The canonical LCEL architecture consists of:
1. A `ChatPromptTemplate` containing a `MessagesPlaceholder(variable_name="history")`.
2. A real Google Gemini Chat Model (`ChatGoogleGenerativeAI`).
3. An in-memory or database session store mapping `session_id -> ChatMessageHistory`.
4. `RunnableWithMessageHistory` wrapping the chain to automatically manage history retrieval and persistence.

In [2]:
# Part 3 Code: Stateful Chatbot with Real Google Gemini & Session Management
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

# 1. Define Prompt Template with MessagesPlaceholder for history injection
chatbot_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful and memory-aware AI study assistant. Answer concisely in 1-2 sentences."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# 2. Real Google Gemini Chat Model
gemini_chat = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    api_key=api_key or "AIzaSy_placeholder_until_env_key_set",
    temperature=0.7
)

# 3. Base Chain
base_chain = chatbot_prompt | gemini_chat | StrOutputParser()

# 4. Session Store: Maps session_id -> InMemoryChatMessageHistory
session_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

# 5. Wrap Base Chain with History Manager
stateful_chatbot = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 6. Execute Multi-Turn Dialog for User Alice (Session: 'alice_123')
alice_config = {"configurable": {"session_id": "alice_123"}}

try:
    print("=" * 60)
    print("👤 USER ALICE — TURN 1:")
    turn1_reply = stateful_chatbot.invoke(
        {"input": "Hi, my name is Alice and I am preparing for PGDCA exams."},
        config=alice_config
    )
    print("AI:", turn1_reply)

    print("\n👤 USER ALICE — TURN 2 (Testing Memory Recall):")
    turn2_reply = stateful_chatbot.invoke(
        {"input": "What is my name and what exam am I preparing for?"},
        config=alice_config
    )
    print("AI:", turn2_reply)

    # 7. Demonstrate Session Isolation with User Bob (Session: 'bob_456')
    bob_config = {"configurable": {"session_id": "bob_456"}}
    print("\n👤 USER BOB — TURN 1 (Isolated Session):")
    bob_turn1_reply = stateful_chatbot.invoke(
        {"input": "Hello! Do you know who I am?"},
        config=bob_config
    )
    print("AI:", bob_turn1_reply)
except Exception as e:
    print("ℹ️ Set your GEMINI_API_KEY in .env to run this stateful chatbot live with Gemini.")
    print(f"Notice: {e}")

👤 USER ALICE — TURN 1:
ℹ️ Set your GEMINI_API_KEY in .env to run this stateful chatbot live with Gemini.
Notice: Error calling model 'gemini-1.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}


### 💡 Theoretical Note: Production Memory Storage Backends
In enterprise production apps, conversations must persist across browser reloads and server restarts. LangChain provides drop-in database backends:

```python
# --- Redis for Ultra-Fast Web Sessions ---
# from langchain_community.chat_message_histories import RedisChatMessageHistory
# history = RedisChatMessageHistory(session_id="user_123", url="redis://localhost:6379")

# --- PostgreSQL for Relational Durability ---
# from langchain_community.chat_message_histories import PostgresChatMessageHistory
# history = PostgresChatMessageHistory(session_id="user_123", connection_string="postgresql://...")
```

## Part 4: Hands-On Student Exercise

**Objective**: Build a simulated student consultation loop with real Google Gemini that feeds a sequence of user questions into the stateful chatbot, tracks conversation progression, and inspects the internal message history state after each turn.

In [3]:
# Student Exercise: Interactive Consultation Loop with Gemini & Memory Inspection
consultation_inputs = [
    "I am planning my capstone project on Healthcare RAG.",
    "Which domain did I choose for my project?",
    "Suggest two Python libraries for this project."
]

student_session_store = {}

def get_student_history(session_id: str):
    if session_id not in student_session_store:
        student_session_store[session_id] = InMemoryChatMessageHistory()
    return student_session_store[session_id]

consultation_bot = RunnableWithMessageHistory(
    base_chain,
    get_student_history,
    input_messages_key="input",
    history_messages_key="history"
)

student_cfg = {"configurable": {"session_id": "student_pgdca_99"}}

print("=" * 60)
print("🎓 STUDENT CONSULTATION SIMULATION WITH REAL GEMINI:")
print("=" * 60)

try:
    for idx, user_query in enumerate(consultation_inputs, 1):
        print(f"\n[Turn {idx}] Student: {user_query}")
        bot_reply = consultation_bot.invoke({"input": user_query}, config=student_cfg)
        print(f"[Turn {idx}] Tutor:   {bot_reply}")

    # Inspect total accumulated history
    final_history = student_session_store["student_pgdca_99"].messages
    print("\n" + "=" * 60)
    print(f"📊 ACCUMULATED CONVERSATION TURNS: {len(final_history)} messages")
    print("=" * 60)
    for msg in final_history:
        role = "Student" if msg.__class__.__name__ == "HumanMessage" else "Tutor"
        print(f"- {role}: {msg.content}")
except Exception as e:
    print("ℹ️ Set your GEMINI_API_KEY in .env to run this simulation live with Gemini.")
    print(f"Notice: {e}")

🎓 STUDENT CONSULTATION SIMULATION WITH REAL GEMINI:

[Turn 1] Student: I am planning my capstone project on Healthcare RAG.
ℹ️ Set your GEMINI_API_KEY in .env to run this simulation live with Gemini.
Notice: Error calling model 'gemini-1.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}
